In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
from tqdm import tqdm
import re

# Optional: install missing libs
# !pip install wordcloud seaborn tqdm

# === 1. Load Data ===
with open("UCFCrime_Train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to dataframe
records = []
for video_name, info in data.items():
    label = video_name.split("_")[0][:-3]  # e.g., "Abuse001_x264" → "Abuse"
    for sent, ts in zip(info["sentences"], info["timestamps"]):
        records.append({
            "video": video_name,
            "label": label,
            "sentence": sent,
            "start": ts[0],
            "end": ts[1],
            "duration": ts[1] - ts[0],
        })

df = pd.DataFrame(records)
print("Total sentences:", len(df))
print(df.head())

# === 2. Basic Statistics ===
print("\n--- Dataset Overview ---")
print(df["label"].value_counts())
print("\nNumber of unique videos:", df["video"].nunique())
print("\nAverage sentence duration per class:")
print(df.groupby("label")["duration"].mean())

# === 3. Class Distribution ===
plt.figure(figsize=(10, 5))
sns.countplot(y="label", data=df, order=df["label"].value_counts().index)
plt.title("Class Distribution")
plt.show()

# === 4. Text Length Analysis ===
df["word_count"] = df["sentence"].apply(lambda x: len(x.split()))
df["char_count"] = df["sentence"].apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df["word_count"], bins=30, kde=True)
plt.title("Distribution of Sentence Lengths (words)")
plt.show()

# === 5. Frequent Words per Class ===
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df["clean_sentence"] = df["sentence"].apply(clean_text)

word_freqs = {}
for label in df["label"].unique():
    words = " ".join(df[df["label"] == label]["clean_sentence"]).split()
    freq = Counter(words)
    word_freqs[label] = freq

# Show top 10 frequent words per class
for label, freq in word_freqs.items():
    print(f"\nTop words for class: {label}")
    print(freq.most_common(10))

# === 6. Word Clouds (optional visualization) ===
for label, freq in word_freqs.items():
    plt.figure(figsize=(8, 6))
    wc = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(freq)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud - {label}")
    plt.show()

# === 7. Co-occurrence Exploration ===
# Check which words appear most across multiple classes
all_words = set()
for label in word_freqs:
    all_words |= set(word_freqs[label].keys())

overlaps = {}
labels = list(word_freqs.keys())
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        l1, l2 = labels[i], labels[j]
        overlap = len(set(word_freqs[l1]).intersection(set(word_freqs[l2])))
        overlaps[(l1, l2)] = overlap

print("\n--- Word overlap between classes ---")
for pair, ov in sorted(overlaps.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(pair, ":", ov)

# === 8. Prepare for Embedding / Clustering ===
# Example: use a simple sentence embedding model
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

model = SentenceTransformer("all-MiniLM-L6-v2")

sample_df = df.sample(min(1000, len(df)), random_state=42)
embeddings = model.encode(sample_df["sentence"].tolist(), show_progress_bar=True)
sample_df["x"] = PCA(n_components=2).fit_transform(embeddings)[:, 0]
sample_df["y"] = PCA(n_components=2).fit_transform(embeddings)[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample_df, x="x", y="y", hue="label", alpha=0.7, palette="tab10")
plt.title("Sentence Embedding Clusters (PCA projection)")
plt.show()

# ==============================================
# 1. Load and aggregate data by video
# ==============================================
import json, re
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Load data
with open("UCFCrime_Train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

records = []
for video_name, info in data.items():
    label = re.sub(r"\d+.*", "", video_name)  # "Abuse001_x264" -> "Abuse"
    sentences = " ".join(info["sentences"])
    duration = info["duration"]
    records.append({
        "video": video_name,
        "label": label,
        "text": sentences,
        "duration": duration,
        "n_sentences": len(info["sentences"])
    })

df = pd.DataFrame(records)
print("Dataset shape:", df.shape)
df.head()

# ==============================================
# 2. Basic dataset statistics
# ==============================================
print("\n--- Class Distribution ---")
print(df["label"].value_counts())

plt.figure(figsize=(10,5))
sns.countplot(y="label", data=df, order=df["label"].value_counts().index)
plt.title("Class Distribution (video-level)")
plt.show()

# Text length analysis
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,5))
sns.histplot(df["word_count"], bins=30, kde=True)
plt.title("Distribution of text length (words per video)")
plt.show()

# ==============================================
# 3. Word frequency per class
# ==============================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text

df["clean_text"] = df["text"].apply(clean_text)

word_freqs = {}
for label in df["label"].unique():
    words = " ".join(df[df["label"] == label]["clean_text"]).split()
    word_freqs[label] = Counter(words)

for label, freq in word_freqs.items():
    print(f"\nTop words for {label}:")
    print(freq.most_common(10))

# Optional: visualize word clouds
from wordcloud import WordCloud
for label, freq in word_freqs.items():
    plt.figure(figsize=(8,6))
    wc = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(freq)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud - {label}")
    plt.show()

# ==============================================
# 4. Embedding Analysis
# ==============================================
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import torch

# Choose embedding model
# You can use violence-specific or general models:
# "jinaai/jina-embeddings-v2-base-en" (strong semantic)
# "all-mpnet-base-v2" (general-purpose)
# "sentence-transformers/LaBSE" (multilingual)
# For violence/abuse detection tasks, contrastive fine-tuning is often done starting from "all-mpnet-base-v2"

model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(df["text"].tolist(), show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings)

# PCA visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

df["x"] = coords[:,0]
df["y"] = coords[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="x", y="y", hue="label", palette="tab10", alpha=0.7)
plt.title("Video-level Embedding Clusters (PCA projection)")
plt.show()

# ==============================================
# 5. Binary label mapping (Normal vs Violent)
# ==============================================
# Simplify to two categories for preliminary contrastive learning
violent_labels = ["Abuse","Fighting","Burglary","Robbery","Assault","Shooting","Arrest","Explosion"]
df["binary_label"] = df["label"].apply(lambda x: "Violent" if x in violent_labels else "Non-Violent")

plt.figure(figsize=(6,4))
sns.countplot(x="binary_label", data=df)
plt.title("Binary label distribution")
plt.show()

# ==============================================
# 6. Simple Contrastive Learning fine-tuning
# ==============================================
from sentence_transformers import losses, InputExample
from torch.utils.data import DataLoader

# Create supervised pairs
train_examples = []
for i, row in df.iterrows():
    train_examples.append(InputExample(texts=[row["text"], row["text"]], label=1.0))  # same sample (positive)
# You can add negative pairs randomly from other classes
for i in range(len(df)):
    j = np.random.randint(0, len(df))
    if df.loc[i, "binary_label"] != df.loc[j, "binary_label"]:
        train_examples.append(InputExample(texts=[df.loc[i, "text"], df.loc[j, "text"]], label=0.0))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.CosineSimilarityLoss(model=model)

# Train a few steps (light fine-tune)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=10,
    show_progress_bar=True
)

# Save fine-tuned model
model.save("violence_contrastive_bert")

# Re-encode for updated clustering visualization
embeddings_tuned = model.encode(df["text"].tolist(), normalize_embeddings=True)
coords_tuned = PCA(n_components=2).fit_transform(embeddings_tuned)
df["x_tuned"], df["y_tuned"] = coords_tuned[:,0], coords_tuned[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="x_tuned", y="y_tuned", hue="binary_label", alpha=0.7, palette="Set2")
plt.title("Embedding space after light contrastive fine-tuning (Violent vs Non-Violent)")
plt.show()

# ==============================================
# 7. Embedding Similarity Heatmaps
# ==============================================
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Compute cosine similarity matrix (tuned embeddings) ---
sim_matrix = cosine_similarity(embeddings_tuned)

# Reorder rows/columns by class for clearer visualization
order = np.argsort(df["binary_label"].values)
sim_sorted = sim_matrix[order][:, order]
labels_sorted = df["binary_label"].iloc[order].values

plt.figure(figsize=(10, 8))
sns.heatmap(sim_sorted, cmap="coolwarm", vmin=-1, vmax=1, cbar_kws={'label': 'Cosine similarity'})
plt.title("Cosine Similarity Heatmap (ordered by binary class)")
plt.xlabel("Samples")
plt.ylabel("Samples")
plt.show()

# ==============================================
# 8. Average Intra- / Inter-class Similarities
# ==============================================
from itertools import combinations

def compute_classwise_similarities(embeddings, labels):
    unique_labels = np.unique(labels)
    results = []
    for l1, l2 in combinations(unique_labels, 2):
        mask1 = labels == l1
        mask2 = labels == l2
        sim = cosine_similarity(embeddings[mask1], embeddings[mask2])
        results.append({
            "pair": f"{l1}-{l2}",
            "mean_sim": sim.mean()
        })
    return pd.DataFrame(results)

# Compute within-class similarity
intra_class = {}
for label in df["binary_label"].unique():
    mask = df["binary_label"] == label
    sim = cosine_similarity(embeddings_tuned[mask])
    intra_class[label] = np.mean(sim[np.triu_indices_from(sim, k=1)])

print("\n--- Average Intra-Class Similarity ---")
for k, v in intra_class.items():
    print(f"{k}: {v:.3f}")

# Compute cross-class similarity
cross_sim = cosine_similarity(
    embeddings_tuned[df["binary_label"] == "Violent"],
    embeddings_tuned[df["binary_label"] == "Non-Violent"]
)
print("\nAverage Cross-Class Similarity:", cross_sim.mean())

# ==============================================
# 9. Multi-Class Visualization (optional)
# ==============================================
# Restrict to a small subset of samples to keep the plot readable
subset_df = df.sample(min(100, len(df)), random_state=42)
subset_emb = embeddings_tuned[subset_df.index]
subset_sim = cosine_similarity(subset_emb)

plt.figure(figsize=(10,8))
sns.heatmap(subset_sim, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Cosine Similarity Heatmap (subset, multi-class)")
plt.xlabel("Sample Index")
plt.ylabel("Sample Index")
plt.show()

# ==============================================
# 10. Cleaned Word Clouds with Stopword Removal (fixed version)
# ==============================================
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from wordcloud import WordCloud
import numpy as np
import matplotlib.pyplot as plt
import re
import pandas as pd

# 1. Define and clean text
custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    'man', 'woman', 'person', 'someone', 'something', 'wearing', 'standing',
    'sitting', 'walking', 'looking', 'left', 'right', 'front', 'back',
    'top', 'shirt', 'pants', 'clothes', 'black', 'white', 'gray', 'fat', 'thin'
})

def clean_and_tokenize(text):
    text = re.sub(r"[^a-z\s]", " ", str(text).lower())
    words = [w for w in text.split() if w not in custom_stopwords and len(w) > 2]
    return " ".join(words)

df["filtered_text"] = df["text"].apply(clean_and_tokenize)

# 2. Compute TF-IDF (use the custom stopwords to be consistent with filtering)
vectorizer = TfidfVectorizer(stop_words=list(custom_stopwords), max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df["filtered_text"])
feature_names = np.array(vectorizer.get_feature_names_out(), dtype=str)

# 3. Aggregate by class label
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
tfidf_df["label"] = df["label"].values

# 4. Compute mean TF-IDF per term per label (coerce non-numeric, skip empty cases)
for label in df["label"].unique():
    sub = tfidf_df.loc[tfidf_df["label"] == label, feature_names]
    if sub.shape[0] == 0:
        # no samples for this label
        continue

    # Ensure all columns are numeric; coerce any weird values to NaN
    sub_numeric = sub.apply(pd.to_numeric, errors="coerce")

    # Compute mean, drop any NaNs
    mean_tfidf = sub_numeric.mean(axis=0).dropna()

    if mean_tfidf.empty or mean_tfidf.sum() == 0:
        # nothing meaningful to show
        print(f"Skipping wordcloud for {label}: no numeric TF-IDF values")
        continue

    top_terms = mean_tfidf.sort_values(ascending=False).head(50)

    plt.figure(figsize=(10,6))
    wc = WordCloud(width=1000, height=500, background_color="white").generate_from_frequencies(top_terms.to_dict())
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"TF-IDF Weighted Word Cloud - {label}")
    plt.show()

# ==============================================
# 13. EMBEDDING GEOMETRY EXPLORATION
# ==============================================
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import seaborn as sns
import matplotlib.pyplot as plt

# Compute embedding norms
df["embedding_norm"] = np.linalg.norm(embeddings_tuned, axis=1)

plt.figure(figsize=(8,5))
sns.histplot(df["embedding_norm"], bins=40, kde=True)
plt.title("Distribution of Embedding Norms")
plt.xlabel("||embedding||₂")
plt.show()

# Compute pairwise cosine similarity statistics
cosine_mat = cosine_similarity(embeddings_tuned)
np.fill_diagonal(cosine_mat, np.nan)  # ignore self-similarity

mean_cosine = np.nanmean(cosine_mat)
std_cosine = np.nanstd(cosine_mat)
print(f"Average cosine similarity (global): {mean_cosine:.3f} ± {std_cosine:.3f}")

# Compute per-class average similarity
class_cosine_stats = []
for label in df["label"].unique():
    mask = df["label"] == label
    sims = cosine_mat[np.ix_(mask, mask)]
    class_cosine_stats.append({
        "label": label,
        "mean_intra_cosine": np.nanmean(sims),
        "std_intra_cosine": np.nanstd(sims)
    })
class_cosine_stats = pd.DataFrame(class_cosine_stats)
print("\nPer-class average intra-cosine:")
print(class_cosine_stats.sort_values("mean_intra_cosine", ascending=False))

# Centroid analysis
# Fix: pass a list of column names (not a tuple) when subsetting
centroids = df.groupby("label")[["x_tuned", "y_tuned"]].mean()
centroid_dist = euclidean_distances(centroids)
centroid_dist_df = pd.DataFrame(centroid_dist, index=centroids.index, columns=centroids.index)

plt.figure(figsize=(8,6))
sns.heatmap(centroid_dist_df, cmap="mako", annot=True, fmt=".2f")
plt.title("Inter-Class Centroid Distances (PCA space)")
plt.show()

# ==============================================
# 14. CLASS IMBALANCE & WEIGHTS
# ==============================================
from sklearn.utils.class_weight import compute_class_weight

labels = df["label"].values
unique_labels = np.unique(labels)
weights = compute_class_weight(class_weight='balanced', classes=unique_labels, y=labels)
class_weights = dict(zip(unique_labels, weights))

print("Class Weights (for balanced training):")
for k, v in class_weights.items():
    print(f"{k:15s}  {v:.3f}")

plt.figure(figsize=(8,5))
sns.barplot(x=list(class_weights.keys()), y=list(class_weights.values()))
plt.title("Computed Class Weights (balanced)")
plt.xticks(rotation=45)
plt.show()

# ==============================================
# 15. MULTI-CLASS SUPERVISED CONTRASTIVE LEARNING (fixed)
# ==============================================
# NOTE: sentence_transformers, losses, InputExample, DataLoader and torch are imported in earlier cells,
# so we avoid re-importing them here.

# Prepare samples: positive pairs must use numeric label (1.0)
examples = [
    InputExample(texts=[text, text], label=1.0)  # positive pair (same sample)
    for text in df["text"]
]

# Create some negative pairs (different labels) labeled 0.0
neg_examples = []
for i in range(len(df)):
    j = np.random.randint(0, len(df))
    if df.loc[i, "label"] != df.loc[j, "label"]:
        neg_examples.append(InputExample(texts=[df.loc[i, "text"], df.loc[j, "text"]], label=0.0))
examples += neg_examples

train_dataloader = DataLoader(examples, shuffle=True, batch_size=8)

# Initialize model (fresh or reuse)
model = SentenceTransformer("all-mpnet-base-v2")

# Use CosineSimilarityLoss for pairwise supervision (1.0 = similar, 0.0 = dissimilar)
# SupConLoss is not available in this environment, so use CosineSimilarityLoss which matches our pairwise labels.
train_loss = losses.CosineSimilarityLoss(model=model)

# Fine-tune (few epochs only)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=10,
    show_progress_bar=True
)

# Save tuned model
model.save("ucf_multiclass_contrastive")

# ==============================================
# 16. LINEAR PROBE CLASSIFICATION
# ==============================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_tuned, df["label"], test_size=0.2, stratify=df["label"], random_state=42
)

clf = LogisticRegression(max_iter=2000, class_weight=class_weights)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(10,8))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=clf.classes_).plot(cmap="Blues", xticks_rotation=45)
plt.title("Linear Probe Confusion Matrix on Embeddings")
plt.show()

# ==============================================
# 17. HARD NEGATIVE MINING — Cross-Class Nearest Neighbors (fixed)
# ==============================================
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure we use a 0..N-1 positional mapping that matches embeddings_tuned order.
# Do not modify the original df; create a positional view.
df_pos = df.reset_index(drop=True)

# Compute cosine similarity matrix between all samples (order matches df_pos)
cosine_mat = cosine_similarity(embeddings_tuned)

# Mask same-class similarities (we only want cross-class)
same_class_mask = np.equal.outer(df_pos["binary_label"].values, df_pos["binary_label"].values)
cosine_mat = cosine_mat.astype(float)  # ensure float so we can set np.nan
cosine_mat[same_class_mask] = np.nan

# For each violent sample (by positional index), find the most similar non-violent one.
hard_negatives = []
for i in range(len(df_pos)):
    row = df_pos.iloc[i]
    if row["binary_label"] != "Violent":
        continue

    similarities = cosine_mat[i]

    # If there are no cross-class candidates (all nan), skip
    if np.all(np.isnan(similarities)):
        continue

    # Get index of the best non-violent (positional index)
    try:
        j = int(np.nanargmax(similarities))
    except ValueError:
        # fallback: skip if argmax cannot be computed
        continue

    # Defensive: if j equals i for some reason, try to pick next best
    if j == i:
        sims_copy = similarities.copy()
        sims_copy[j] = np.nan
        if np.all(np.isnan(sims_copy)):
            continue
        j = int(np.nanargmax(sims_copy))

    hard_negatives.append({
        "violent_pos_idx": i,
        "violent_video": row["video"],
        "violent_label": row["label"],
        "violent_text": (row["text"][:250] + "...") if isinstance(row["text"], str) else row["text"],
        "closest_nonviolent_pos_idx": j,
        "closest_nonviolent_video": df_pos.loc[j, "video"],
        "closest_nonviolent_text": (df_pos.loc[j, "text"][:250] + "...") if isinstance(df_pos.loc[j, "text"], str) else df_pos.loc[j, "text"],
        "similarity": float(similarities[j])
    })

hard_df = pd.DataFrame(hard_negatives).sort_values("similarity", ascending=False)
print("Top Hard Negatives (Violent ↔ Non-Violent pairs):")
if not hard_df.empty:
    display(hard_df.head(5))
else:
    print("No hard negatives found.")

# Save them for manual inspection
if not hard_df.empty:
    hard_df.to_csv("hard_negatives_pairs.csv", index=False)
    print("Saved to hard_negatives_pairs.csv")
else:
    print("Skipping save: no pairs to save.")

# ==============================================
# 17b. VISUAL SUMMARY
# ==============================================
if not hard_df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(hard_df["similarity"], bins=40, kde=True, color="tomato")
    plt.title("Distribution of Hard Negative Similarities (Violent→Non-Violent)")
    plt.xlabel("Cosine similarity")
    plt.show()

    # Compute average "hardness" per violent class
    avg_sim_per_class = hard_df.groupby("violent_label")["similarity"].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,5))
    sns.barplot(x=avg_sim_per_class.index, y=avg_sim_per_class.values, palette="Reds_r")
    plt.title("Average Hard Negative Similarity per Violent Class")
    plt.ylabel("Mean similarity to Non-Violent samples")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No visual summary generated (no hard negatives).")

# ==============================================
# 18. HARD NEGATIVE LINKS IN UMAP SPACE (optional)
# ==============================================
# Ensure UMAP is available, then compute 2D UMAP coordinates from embeddings_tuned
import importlib
if importlib.util.find_spec("umap") is None:
    # install umap-learn if missing
    %pip install --quiet umap-learn

# Use the stable import path for different umap-learn versions
try:
    from umap.umap_ import UMAP
except Exception:
    try:
        import umap
        UMAP = umap.UMAP  # fallback for some package builds
    except Exception as e:
        raise ImportError("Failed to import UMAP from umap-learn. Please install umap-learn.") from e

# Compute UMAP only if coordinates are not already present
if "umap_x" not in df.columns or "umap_y" not in df.columns:
    reducer = UMAP(n_components=2, random_state=42)
    umap_emb = reducer.fit_transform(embeddings_tuned)
    df["umap_x"] = umap_emb[:, 0]
    df["umap_y"] = umap_emb[:, 1]

subset_pairs = hard_df.head(50)  # top 50 hardest
plt.figure(figsize=(9,7))

sns.scatterplot(
    data=df, x="umap_x", y="umap_y",
    hue="binary_label", alpha=0.6, palette={"Violent": "red", "Non-Violent": "blue"}
)

for _, row in subset_pairs.iterrows():
    # find positional indices safely (skip if not found)
    matches_v = df.index[df["video"] == row["violent_video"]]
    matches_nv = df.index[df["video"] == row["closest_nonviolent_video"]]
    if len(matches_v) == 0 or len(matches_nv) == 0:
        continue
    v_idx = matches_v[0]
    nv_idx = matches_nv[0]

    plt.plot(
        [df.loc[v_idx, "umap_x"], df.loc[nv_idx, "umap_x"]],
        [df.loc[v_idx, "umap_y"], df.loc[nv_idx, "umap_y"]],
        color="gray", linewidth=0.7, alpha=0.4
    )

plt.title("Hard Negative Pairs (Violent–Non-Violent links in UMAP space)")
plt.show()

# ==============================================
# 19. HARD-NEGATIVE CONTRASTIVE FINE-TUNING LOOP
# ==============================================
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
import numpy as np
import torch
import random

# Step 1 — Start from the previous tuned model or base
model = SentenceTransformer("all-mpnet-base-v2")

# Step 2 — Build training examples
examples = []

# Positive pairs (same text)
for text, label in zip(df["text"], df["binary_label"]):
    examples.append(InputExample(texts=[text, text], label=1.0))

# Negative pairs (random across opposite labels)
violents = df[df["binary_label"] == "Violent"]
non_violents = df[df["binary_label"] == "Non-Violent"]

for i in range(len(violents)):
    nv_text = non_violents.sample(1, random_state=i).iloc[0]["text"]
    examples.append(InputExample(
        texts=[violents.iloc[i]["text"], nv_text],
        label=0.0
    ))

# Step 3 — Add HARD negatives mined earlier
# (these are the “almost confusing” pairs)
for _, r in hard_df.iterrows():
    examples.append(InputExample(
        texts=[r["violent_text"], r["closest_nonviolent_text"]],
        label=0.0
    ))

print(f"Total pairs for contrastive training: {len(examples)}")

# Step 4 — Build dataloader
train_dataloader = DataLoader(examples, shuffle=True, batch_size=8)

# Step 5 — Define loss
train_loss = losses.CosineSimilarityLoss(model=model)

# Step 6 — Fine-tune
epochs = 2
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=epochs,
    warmup_steps=10,
    show_progress_bar=True
)

# Step 7 — Save the model
# model.save("ucf_hardneg_contrastive")
# print("Model saved to 'ucf_hardneg_contrastive'")

# ==============================================
# 20. EVALUATE AFTER HARD-NEGATIVE TRAINING
# ==============================================
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import seaborn as sns
import matplotlib.pyplot as plt

# Re-encode with updated model
embeddings_new = model.encode(df["text"].tolist(), normalize_embeddings=True)

# Quick projection
coords_new = PCA(n_components=2).fit_transform(embeddings_new)
df["x_new"], df["y_new"] = coords_new[:,0], coords_new[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="x_new", y="y_new", hue="binary_label", alpha=0.8)
plt.title("Post-Hard-Negative Contrastive Embeddings (PCA)")
plt.show()

# Compute metrics again
y_bin = LabelEncoder().fit_transform(df["binary_label"])
sil_new = silhouette_score(embeddings_new, y_bin)
cal_new = calinski_harabasz_score(embeddings_new, y_bin)
dav_new = davies_bouldin_score(embeddings_new, y_bin)

print(f"\nNew Separability Metrics (Binary)")
print(f"Silhouette: {sil_new:.3f}")
print(f"Calinski–Harabasz: {cal_new:.1f}")
print(f"Davies–Bouldin: {dav_new:.3f}")

